In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
# Initialize dataset for Train
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import os
from torch.utils.data import Dataset
from PIL import Image
import glob
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images
    transforms.RandomRotation(10),  # Rotate images randomly within ±15 degrees # Adjust brightness randomly
    transforms.ToTensor() # Convert to tensor
])

transform_valid_test = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images  # Rotate images randomly within ±15 degrees # Adjust brightness randomly
    transforms.ToTensor() # Convert to tensor
])


train_path = os.path.join(path, "PlantVillage", "train")
test_path = os.path.join(path, "PlantVillage", "test")

train_dataset = ImageFolder(train_path, transform=transform)                                                    ## Replaced SkinCancerDataset with ImageFolder
test_dataset = ImageFolder(test_path, transform=transform_valid_test)

batch_size = 12
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)



import matplotlib.pyplot as plt
import numpy as np

# Get a batch of training data
data_iter = iter(train_loader)
images, labels = next(data_iter)

# CIFAR-10 class names
classes = ['Early', "late", "healthy"]

# Show images
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    img = images[i]
    img = np.transpose(img.numpy(), (1, 2, 0))  # Convert (C, H, W) to (H, W, C)

    ax.imshow(img)
    ax.set_title(classes[labels[i].item()])
    ax.axis("off")

plt.show()

In [ ]:
# Write your code here
import torch.nn as nn
import torch

class CNNModel(nn.Module):
    def __init__(self, num_classes=3):
        super(CNNModel, self).__init__()
        self.Con1 = nn.Sequential(
            # Input: 3 x 32 x 32
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=2),
            nn.BatchNorm2d(16),# Output: 96 x 16 x 16
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=1),                  # Output: 96 x 7 x 7
        )
        self.Con2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),# Output: 256 x 7 x 7
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=1),                  # Output: 256 x 3 x 3
        )
        self.Con3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3),
            nn.BatchNorm2d(64),# Output: 384 x 3 x 3
            nn.ReLU(inplace=True),
        )
        self.Con4 = nn.Sequential(
            nn.Conv2d(64, 96, kernel_size=3),
            nn.BatchNorm2d(96),# Output: 384 x 3 x 3
            nn.ReLU(inplace=True),
        )
        self.Con5 = nn.Sequential(
            nn.Conv2d(96, 128, kernel_size=3),
            nn.BatchNorm2d(128),# Output: 256 x 3 x 3
            nn.ReLU(inplace=True),                 # Output: 256 x 2 x 2
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 9 * 9, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.Con1(x)
        x = self.Con2(x)
        x = self.Con3(x)
        x = self.Con4(x)
        x = self.Con5(x)
        # print(x.shape)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [ ]:
# Write your code here
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


In [ ]:
# Write your code here
import torch.optim as optim

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNModel().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.AdamW(model.parameters(), lr=0.0001)  # Adam optimizer
num_epochs = 10 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")

import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()

In [ ]:
# Write your code here
# Write your code here
import torch.nn as nn
import torch

class CNNModel(nn.Module):
    def __init__(self, num_classes=3):
        super(CNNModel, self).__init__()
        self.Con1 = nn.Sequential(
            # Input: 3 x 32 x 32
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=2),
            nn.BatchNorm2d(16),# Output: 96 x 16 x 16
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=1),                  # Output: 96 x 7 x 7
        )
        self.Con2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),# Output: 256 x 7 x 7
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=1),                  # Output: 256 x 3 x 3
        )
        self.Con3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3),
            nn.BatchNorm2d(64),# Output: 384 x 3 x 3
            nn.ReLU(inplace=True),
        )
        self.Con4 = nn.Sequential(
            nn.Conv2d(64, 96, kernel_size=3),
            nn.BatchNorm2d(96),# Output: 384 x 3 x 3
            nn.ReLU(inplace=True),
        )
        self.Con5 = nn.Sequential(
            nn.Conv2d(96, 128, kernel_size=3),
            nn.BatchNorm2d(128),# Output: 256 x 3 x 3
            nn.ReLU(inplace=True),                 # Output: 256 x 2 x 2
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 9 * 9, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes)
        )


        # for the Residual
        self.pool = nn.MaxPool2d(kernel_size=2, stride=1)
        self.conv = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)   #  p = (f-1) / 2 = 2/2 = 1
    def forward(self, x):
        x1 = self.Con1(x)
        x2 = self.Con2(x1)
        x3 = self.Con3(x2)

        # Residual
        # print(x2.shape)
        x2 = self.pool(x2)
        x2 = self.pool(x2)
        x2 = self.conv(x2)
        # print(x2.shape)
        # print(x3.shape)

        x4 = self.Con4(x3 + x2)
        x5 = self.Con5(x4)
        # print(x.shape)
        x6 = torch.flatten(x5, 1)
        x7 = self.classifier(x6)
        return x7

In [ ]:
# Write your code here
import torch.optim as optim

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNModel().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.AdamW(model.parameters(), lr=0.0001)  # Adam optimizer
num_epochs = 10 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")

import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()